# Climate reanalysis — ERA5-Land monthly 2 m temperature

ERA5-Land monthly aggregates resampled to a yearly mean 2 m temperature over the Nile delta — a typical climate-reanalysis pipeline.

## Setup

First the imports. `pyramids` provides `Dataset` (GeoTIFF/NetCDF reading); `earthlens` provides
the unified `EarthLens` entry point and the bundled GEE `Catalog`.

In [ ]:
import os
from pathlib import Path

from pyramids.dataset import Dataset
from pyramids.plot import ColorBar

from earthlens.core import EarthLens
from earthlens.gee import Catalog, cancel_task

### Output directory and credentials

Written GeoTIFFs go under a per-notebook `out/` directory. The GEE service-account credentials
are read from the `GEE_SERVICE_ACCOUNT` / `GEE_SERVICE_KEY` environment variables — both must be
set before running this cell.

In [ ]:
OUT_DIR = Path('out') / 'climate-reanalysis'
OUT_DIR.mkdir(parents=True, exist_ok=True)

SERVICE_ACCOUNT = os.environ['GEE_SERVICE_ACCOUNT']
SERVICE_KEY = os.environ['GEE_SERVICE_KEY']
print(f'output directory: {OUT_DIR.resolve()}')

## Inspect the catalog entry

Before downloading anything, look at what the bundled catalog knows about the asset — bands, cadence, license, provider.

In [ ]:
cat = Catalog()
ds = cat.get_dataset('ECMWF/ERA5_LAND/MONTHLY_AGGR')
print(ds)

# The summary clips long text and shows only a band count, so an explorer
# notebook still wants the untruncated title and the fields it omits:
print(f'title (full):        {ds.title}')
print(f'ee_type:             {ds.ee_type}')
print(f'default_reducer:     {ds.default_reducer}')
print(f'license:             {ds.license}')
print(f'band ids (first 5):  {list(ds.bands)[:5]}')

## Download

Tiny AOI ([29.0, 32.0] lat, [30.0, 33.0] lon) at 11132.0 m, `yearly` cadence — keeps the synchronous download under EE's 32768-px per-axis cap.

### Build the request

Construct the `EarthLens` request first — source, dataset, band (`variables`), AOI, date window,
cadence, and the synchronous `getDownloadURL` defaults.

In [ ]:
gee = EarthLens(
    data_source="gee",
    start='2023-01-01',
    end='2023-12-31',
    dataset='ECMWF/ERA5_LAND/MONTHLY_AGGR',
    variables=['temperature_2m'],
    aoi=[30.0, 29.0, 33.0, 32.0],
    cadence='yearly',
    path=OUT_DIR,
    scale=11132.0,
    reducer='mean',
)

### Authenticate and download

`authenticate()` resolves the service-account credentials on its own line, then `download()` fetches the
composite to disk. A live Earth Engine failure surfaces here rather than being swallowed.

In [ ]:
gee.authenticate(service_account=SERVICE_ACCOUNT, service_key=SERVICE_KEY)
paths = gee.download(progress_bar=False)
print(f'wrote {len(paths)} GeoTIFF(s):')
for p in paths:
    print(f'  {p}  ({p.stat().st_size / 1024:.1f} KB)')

## Quick preview

Load the first written GeoTIFF through pyramids and render the single band. (`pyramids.dataset.Dataset` is the project's GeoTIFF/NetCDF wrapper.)

### Load and declare the fill

Read the first written GeoTIFF through pyramids. The band arrives declaring no nodata: the EEDAI reader fills
the pixels Earth Engine had masked — ERA5-Land is a *land* reanalysis, so the Mediterranean and the Red Sea are
masked — but never stamps the fill into the band. The fill here is `0`, which as a 2 m air temperature in
kelvin is physically impossible, so it is unambiguous: declare it and the sea drops out of both the map and the
statistics.

In [ ]:
preview = Dataset.read_file(paths[0], read_only=False)
print(f'dtype        : {preview.dtype}')
print(f'declared     : {preview.no_data_value}')

preview.no_data_value = [0.0]
print(f'after declare: {preview.no_data_value}')

### Render the band

Display the single-band array and report its value range.

In [ ]:
glyph = preview.plot(
    cmap='inferno',
    colorbar=ColorBar(label='2 m air temperature (K)'),
    title='ERA5-Land 2 m temperature — 2023 mean',
)
glyph.ax.title.set_fontsize(11)

stats = preview.stats(approx_ok=False)
low, high = stats['min'].iloc[0], stats['max'].iloc[0]
print(f'value range: [{low:.4g}, {high:.4g}] K')
# 0 K is the fill, not a temperature. If the declaration had not taken effect
# the minimum would still be 0, so assert a physically possible surface value.
assert low > 100, f'2 m temperature should be well above 100 K after masking; got {low}'

land = preview.count_domain_cells()
total_cells = preview.rows * preview.columns
print(f'land cells:  {land:,} of {total_cells:,}')

# The handle was opened writable to declare the nodata; release it so the
# GDAL lock does not outlive the cell.
preview.close()

## Tracking submitted jobs (asynchronous export)

The download above uses `export_via="url"` — a synchronous `getDownloadURL` round-trip. Nothing was queued, so there's no Earth Engine job to track.

To track an export instead, switch to an asynchronous sink (`drive` / `gcs` / `asset`) and pass `wait_for_export=False` so `.download()` returns a `TaskInfo` at submission time rather than blocking until completion. The cells below submit the same `(asset_id, band, AOI, scale)` request as an `export_via="asset"` task into the service account's own asset folder, then walk the four jobs-API calls (`list_recent_tasks` → `wait_for_task_id` → `ee.data.getAsset` → `ee.data.deleteAsset`) to make the job finish *and* tidy up. See `track-batch-exports.ipynb` for a deeper worked example.

### Imports and the demo asset folder

Bring in `ee` plus the `earthlens.gee` task helpers, and derive a `Folder` asset path under the
current project. The backend writes the image at `<asset_id>/<prefix>`, so this `asset_id` is the
parent folder, not the final image path.

In [ ]:
import ee

from earthlens.gee import list_recent_tasks, wait_for_task_id

# The asset goes into a `Folder` asset that we own. `GEE._export_via_batch`
# writes the actual image at `<asset_id>/<prefix>`, so `asset_id` here is
# the parent FOLDER (not the final image path). Both must be cleaned up.
_proj = ee.data._get_projects_path().removeprefix('projects/')
PARENT = f'projects/{_proj}/assets'
DEMO_FOLDER = f'{PARENT}/earthlens-demo-climate-reanalysis'
print(f'demo folder: {DEMO_FOLDER}')

### Prepare a clean folder

Listing the parent says whether a previous run left the folder behind, so the cleanup never has to swallow a
"not found" from Earth Engine. Then create the folder Earth Engine requires before a child write.

In [ ]:
# Listing the parent says whether a previous run left the folder behind, so the
# cleanup below never has to swallow a "not found" from Earth Engine.
listed = ee.data.listAssets({'parent': PARENT})
siblings = [asset['name'] for asset in listed.get('assets', [])]
if DEMO_FOLDER in siblings:
    children = ee.data.listAssets({'parent': DEMO_FOLDER})
    for child in children.get('assets', []):
        ee.data.deleteAsset(child['name'])
        print(f'cleared leftover child: {child["name"]}')
    ee.data.deleteAsset(DEMO_FOLDER)
    print(f'cleared leftover folder: {DEMO_FOLDER}')
# Create the parent folder — EE requires it to exist before a child write.
ee.data.createAsset({'type': 'Folder'}, DEMO_FOLDER)
print(f'created folder: {DEMO_FOLDER}')

### Submit

Same `(asset_id, band, AOI, scale)` request as the sync download above, just routed through `export_via="asset"` + `wait_for_export=False`. `download()` returns a `TaskInfo` per submitted bucket at the moment the task is queued — no blocking.

### Build the async request and authenticate

Same `(asset_id, band, AOI, scale)` request as the sync download, but routed through
`export_via="asset"` + `wait_for_export=False`. Construct it first, then `authenticate()` on its
own line.

In [ ]:
async_gee = EarthLens(
    data_source="gee",
    start='2023-01-01',
    end='2023-12-31',
    dataset='ECMWF/ERA5_LAND/MONTHLY_AGGR',
    variables=['temperature_2m'],
    aoi=[30.0, 29.0, 33.0, 32.0],
    cadence='yearly',
    path=OUT_DIR,
    scale=11132.0,
    reducer='mean',
    export_via='asset',
    asset_id=DEMO_FOLDER,
    wait_for_export=False,
)
async_gee.authenticate(service_account=SERVICE_ACCOUNT, service_key=SERVICE_KEY)

### Submit the export

`download()` returns a `TaskInfo` per submitted bucket at the moment the task is queued — no blocking.

In [ ]:
submitted = async_gee.download(progress_bar=False)
task_info = submitted[0]
print(f'submitted: id={task_info.id} state={task_info.state}')
print(f'           description={task_info.description}')

### List + wait

`list_recent_tasks(description_prefix=...)` returns every matching task across the current project; `wait_for_task_id` blocks until the one we care about reaches a terminal state. A real workflow would just poll later from a separate process — the wait here exists so the notebook shows the full success path end-to-end.

In [ ]:
recent = list_recent_tasks(
    description_prefix=task_info.description,
    max_age_min=10,
)
print(f'list_recent_tasks matched {len(recent)} task(s):')
for t in recent:
    print(f'  {t.id}  {t.state:<12} {t.description}')
final = None
try:
    final = wait_for_task_id(
        task_info.id,
        poll_seconds=10,
        progress_bar=False,
    )
    print(f'\nfinal state: {final.state}')
finally:
    if final is None:
        # The wait raises on FAILED / CANCELLED *and on timeout* — and a
        # timeout leaves the export still running. Cancel it so an aborted
        # notebook does not leave a live task behind; cancel_task is a no-op
        # on an already-terminal task, and the original error still
        # propagates out of this finally.
        cancel_task(task_info.id)
        print(f'cancelled {task_info.id} after the wait failed')

### Verify + clean up

Confirm the produced asset exists on Earth Engine, then delete it (and the surrounding demo folder) so we don't leak storage between notebook runs. The backend wrote the image at `<DEMO_FOLDER>/<task description>`.

In [ ]:
produced = f'{DEMO_FOLDER}/{task_info.description}'
meta = ee.data.getAsset(produced)
print(f'asset exists: type={meta.get("type")} name={meta.get("name")}')
ee.data.deleteAsset(produced)
print('asset deleted')
# Tear down the parent folder.
ee.data.deleteAsset(DEMO_FOLDER)
print(f'folder deleted: {DEMO_FOLDER}')

## What's on disk

The written GeoTIFF is left under the per-notebook `out/` directory for you to inspect. That directory is
`.gitignore`d — re-running the notebook overwrites it.

In [ ]:
for p in sorted(OUT_DIR.iterdir()) if OUT_DIR.exists() else []:
    print(f'{p}  ({p.stat().st_size / 1024:.1f} KB)')